# Final results (after kfold hyperparam. selection)

### Load results from wandb

In [6]:
%load_ext autoreload
%autoreload 2
import sys, os

# Get the parent directory (TopoProteo/)
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
# Add it to Python’s import path
sys.path.append(parent_dir)

import numpy as np
from utils_table_generator import *
from utils_final_results import *
from proteo.evaluation_clean import ModelLoader

wandb_username = "lcornelis"  # Change this to your W&B username if needed
wandb_project = "ProteoFinalAll_test"  # Change this to your W&B project name if needed
metric = "mse"  # Change this to the metric you want to extract (e.g., "mae", "mse", etc.)
original_units = True  # Set to True if you want to convert back to original units
csv_filename = "final_results_test.csv"  # Output CSV filename
save_csv = False  # Set to True if you want to save the grouped results to a CSV file

df = load_results_dataframe(wandb_username, wandb_project, original_units=original_units, metric=metric, csv_filename=csv_filename, save_csv=save_csv)
# df, grouped, best_configs, summary, table = generate_table(df, save_csv=save_csv, csv_filename=csv_filename)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
▶ Number of runs fetched from W&B: 15
▶ After building df, df.shape = (15, 184)


### Filter dataframe to see final performances and checkpoints

In [7]:
columns_to_keep = ["dataset", "model", "test_mae", "checkpoint", "best_epoch/checkpoint"]
print([col for col in df.columns if "checkpoint" in col])

filtered_df = df.reindex(columns=columns_to_keep)
pd.set_option('display.max_colwidth', None)
filtered_df

['checkpoint', 'best_epoch/checkpoint', 'callbacks.model_checkpoint.mode', 'callbacks.model_checkpoint.dirpath', 'callbacks.model_checkpoint.monitor', 'callbacks.model_checkpoint.verbose', 'callbacks.model_checkpoint._target_', 'callbacks.model_checkpoint.filename', 'callbacks.model_checkpoint.save_last', 'callbacks.model_checkpoint.save_top_k', 'callbacks.model_checkpoint.every_n_epochs', 'callbacks.model_checkpoint.save_weights_only', 'callbacks.model_checkpoint.every_n_train_steps', 'callbacks.model_checkpoint.train_time_interval', 'callbacks.model_checkpoint.auto_insert_metric_name', 'callbacks.model_checkpoint.save_on_train_epoch_end']


,dataset,model,test_mae,checkpoint,best_epoch/checkpoint
0,pointcloud,mlp,0.103512,/scratch/lcornelis/outputs/checkpoints/epoch_024-v931.ckpt,None
1,wgcna,gcn,0.136756,/scratch/lcornelis/outputs/checkpoints/epoch_185-v108.ckpt,None
2,wgcna,gat,0.121351,/scratch/lcornelis/outputs/checkpoints/epoch_036-v866.ckpt,None
3,pointcloud,mlp,0.079689,/scratch/lcornelis/outputs/checkpoints/epoch_137-v224.ckpt,None
4,pointcloud,mlp,0.103512,/scratch/lcornelis/outputs/checkpoints/epoch_024-v934.ckpt,None
5,pointcloud,mlp,0.103512,None,/scratch/lcornelis/outputs/checkpoints/epoch_024-v935.ckpt
6,pointcloud,mlp,0.079689,None,/scratch/lcornelis/outputs/checkpoints/epoch_137-v225.ckpt
7,wgcna,gcn,0.136756,None,/scratch/lcornelis/outputs/checkpoints/epoch_185-v109.ckpt
8,wgcna,gcn,0.131820,None,/scratch/lcornelis/outputs/checkpoints/epoch_167-v104.ckpt
9,wgcna,gat,0.121351,None,/scratch/lcornelis/outputs/checkpoints/epoch_036-v867.ckpt


### Load the corresponding config file and checkpoint

In [6]:
model, config = ModelLoader.load_checkpoint("/scratch/lcornelis/outputs/checkpoints/epoch_142-v127.ckpt")
print(type(model))


<class 'topobench.model.model.TBModel'>


Here we assume a single run per (model, adj_metric) pair

### Load dataset

In [ ]:
adj_metric = "spearman_correlation"
adj_threshold = 0.5
kfold = True
num_folds = 5
fold = 0

train_dataset, val_dataset, _ = load_dataset(adj_metric, adj_threshold, kfold=kfold, num_folds=num_folds, fold=fold)

Processed file names: ['FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt', 'FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt
Processed file names: ['FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt', 'FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_nfl_spearman_correlation_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_r

### Run checkpoint on sporadic dataset